# Etapa 3B — ¿El predictor identifica la dinámica?

Entrenamos la misma arquitectura sobre tres condiciones con marginales de observación idénticas: estática, cíclica e independiente. Sólo cambia qué ventana futura se empareja con cada ventana actual.

La pregunta principal no es cuánto baja la loss, sino qué operador explica mejor la acción del predictor aprendido:

$$E_d(M,A)=\frac{\lVert MA-AK_d\rVert_F}{\lVert A\rVert_F}.$$

Para cada modelo comparamos los tres candidatos. La clasificación por acción y la clasificación por espectro deben acertar al menos 8/10 seeds en cada condición. El protocolo completo fue congelado en `docs/STAGE3_THREE_DYNAMICS_PROTOCOL.md` antes de ejecutar este notebook.

In [ ]:
# ruff: noqa: E402, E501, I001
import json
import sys
from math import comb
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import yaml
from IPython.display import Markdown, display

from koopman_jepa.config import DataConfig, ExperimentConfig, ModelConfig, TrainConfig, validate_config
from koopman_jepa.model import TemporalJEPA
from koopman_jepa.phase_analysis import evaluate_phase_operator_candidates, evaluate_phase_operator_diagnostics, evaluate_phase_representation
from koopman_jepa.phase_data import PhaseWindowConfig, make_phase_tensor_dataset_splits
from koopman_jepa.training import collect_paired_embeddings, select_device, set_seed, train_model

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
CONFIG_PATH = ROOT / "configs" / "stage3_three_dynamics_control.yaml"
with CONFIG_PATH.open(encoding="utf-8") as handle:
    raw = yaml.safe_load(handle)

dynamics = tuple(raw["dynamics"])
seeds = tuple(raw["seeds"])
candidates = tuple(raw["operator_identification"]["candidates"])
minimum_correct = raw["operator_identification"]["minimum_correct_seeds_per_dynamics"]
chance_probability = raw["operator_identification"]["chance_probability"]
rollout_horizons = tuple(raw["diagnostics"]["rollout_horizons"])
required_active_rank = raw["diagnostics"]["required_active_rank"]
assert dynamics == candidates == ("static", "cyclic", "independent")
assert seeds == tuple(range(1, 11))
null_tail = sum(comb(len(seeds), k) * chance_probability**k * (1.0 - chance_probability)**(len(seeds) - k) for k in range(minimum_correct, len(seeds) + 1))
assert np.isclose(null_tail, raw["operator_identification"]["one_sided_tail_probability_at_threshold"])
emission_config = PhaseWindowConfig(**raw["emission"], repeats_per_transition=1)
model_raw = raw["model"]
print(json.dumps({"config": CONFIG_PATH.name, "dynamics": dynamics, "seeds": seeds, "minimum_correct": minimum_correct, "null_tail": null_tail, "test_constructed": False}, indent=2))

## Diseño de la comparación

Dentro de cada seed reiniciamos los tres modelos con los mismos pesos. Los bancos de ventanas actuales y futuras son los mismos; únicamente cambia el pairing temporal. La matriz de alineación $A$ se estima con las fases de train y se usa para comparar la acción de $M$ contra los tres operadores candidatos.

Una seed sin rango activo 3 se cuenta como no identificada: no podemos declarar un espectro tridimensional si la representación descartó algún modo.

In [ ]:
def make_experiment_config(seed):
    config = ExperimentConfig(
        data=DataConfig(context_length=emission_config.window_length),
        model=ModelConfig(latent_dim=model_raw["latent_dim"], channels=model_raw["channels"], predictor_init=model_raw["predictor_init"]),
        train=TrainConfig(seed=seed, **raw["train"]),
    )
    validate_config(config)
    return config

def make_model(device):
    return TemporalJEPA(
        latent_dim=model_raw["latent_dim"],
        channels=model_raw["channels"],
        predictor_init=model_raw["predictor_init"],
        pooling=model_raw["pooling"],
        input_length=emission_config.window_length,
    ).to(device)

def evaluate_run(model, train_dataset, validation_dataset, config, dynamics_name, seed, history):
    device = select_device(config.train.device)
    train_current, _, _, train_phase_pairs = collect_paired_embeddings(model, train_dataset, config.train.batch_size, device)
    val_current, val_future_online, val_future_target, val_phase_pairs = collect_paired_embeddings(model, validation_dataset, config.train.batch_size, device)
    predictor = model.predictor.matrix.detach().cpu().numpy()
    representation = evaluate_phase_representation(
        train_current, train_phase_pairs[:, 0], val_current, val_phase_pairs[:, 0], predictor, dynamics_name, seed
    )
    comparison = evaluate_phase_operator_candidates(
        train_current, train_phase_pairs[:, 0], predictor, candidates, rollout_horizons
    )
    diagnostic = evaluate_phase_operator_diagnostics(
        val_current, val_future_online, val_future_target, val_phase_pairs[:, 0], val_phase_pairs[:, 1], predictor, dynamics_name
    )
    full_rank = comparison["active_rank"] == required_active_rank
    spectral_errors = comparison["spectral_mean_errors"]
    return {
        "seed": seed,
        "dynamics": dynamics_name,
        "full_rank": full_rank,
        "action_correct": full_rank and comparison["predicted_action_dynamics"] == dynamics_name,
        "spectrum_correct": full_rank and comparison["predicted_spectral_dynamics"] == dynamics_name,
        "predicted_action": comparison["predicted_action_dynamics"],
        "predicted_spectrum": comparison["predicted_spectral_dynamics"],
        "action_errors": comparison["action_errors"],
        "spectral_errors": spectral_errors,
        "action_margin": comparison["action_margin"],
        "spectrum_margin": comparison["spectrum_margin"],
        "rollout_errors": comparison["rollout_errors"],
        "active_eigenvalues": comparison["active_eigenvalues"],
        "effective_rank": representation["effective_rank"],
        "phase_probe_accuracy": representation["linear_probe_accuracy"],
        "phase_alignment_error": representation["phase_alignment_error"],
        "active_invariance_error": representation["active_invariance_error"],
        "predictor_vs_posthoc_online_error": diagnostic["predictor_vs_posthoc_online_error"],
        "final_prediction_loss": history[-1]["val_prediction_loss"],
        "final_total_loss": history[-1]["val_loss"],
    }

In [ ]:
results = []
histories = {}
for seed in seeds:
    config = make_experiment_config(seed)
    splits = make_phase_tensor_dataset_splits(
        emission_config,
        train_repeats_per_transition=raw["splits"]["train_repeats_per_transition"],
        validation_repeats_per_transition=raw["splits"]["validation_repeats_per_transition"],
        seed=seed,
    )
    assert splits.test is None
    assert len({len(splits.train[name]) for name in dynamics}) == 1
    assert len({len(splits.validation[name]) for name in dynamics}) == 1
    for dynamics_name in dynamics:
        set_seed(seed)
        device = select_device(config.train.device)
        model = make_model(device)
        history = train_model(model, splits.train[dynamics_name], splits.validation[dynamics_name], config, device)
        assert all(np.isfinite(value) for row in history for value in row.values())
        histories[(seed, dynamics_name)] = history
        result = evaluate_run(model, splits.train[dynamics_name], splits.validation[dynamics_name], config, dynamics_name, seed, history)
        results.append(result)
        print(f"seed={seed:02d} true={dynamics_name:11s} action={result['predicted_action']:11s} spectrum={str(result['predicted_spectrum']):11s} rank3={result['full_rank']}")

assert len(results) == len(seeds) * len(dynamics)

In [ ]:
summary = {}
for dynamics_name in dynamics:
    subset = [row for row in results if row["dynamics"] == dynamics_name]
    action_correct = sum(row["action_correct"] for row in subset)
    spectrum_correct = sum(row["spectrum_correct"] for row in subset)
    full_rank = sum(row["full_rank"] for row in subset)
    summary[dynamics_name] = {
        "full_rank_seeds": full_rank,
        "correct_action_seeds": action_correct,
        "correct_spectrum_seeds": spectrum_correct,
        "median_correct_action_error": float(np.median([row["action_errors"][dynamics_name] for row in subset])),
        "median_action_margin": float(np.median([row["action_margin"] for row in subset])),
        "median_spectrum_margin": float(np.nanmedian([row["spectrum_margin"] if row["spectrum_margin"] is not None else np.nan for row in subset])),
        "median_effective_rank": float(np.median([row["effective_rank"] for row in subset])),
        "median_phase_probe_accuracy": float(np.median([row["phase_probe_accuracy"] for row in subset])),
        "operator_identification_passed": action_correct >= minimum_correct and spectrum_correct >= minimum_correct,
    }
global_operator_result = all(values["operator_identification_passed"] for values in summary.values())
display(Markdown("## Resumen agregado"))
print(json.dumps({"conditions": summary, "global_operator_result": global_operator_result, "loss_used_for_decision": False}, indent=2))

In [ ]:
action_matrix = np.array([[np.median([row["action_errors"][candidate] for row in results if row["dynamics"] == truth]) for candidate in candidates] for truth in dynamics])
spectral_matrix = np.array([[np.nanmedian([row["spectral_errors"][candidate] if row["spectral_errors"] is not None else np.nan for row in results if row["dynamics"] == truth]) for candidate in candidates] for truth in dynamics])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for axis, matrix, title in zip(axes, (action_matrix, spectral_matrix), ("Error mediano de acción $MA-AK_d$", "Error espectral mediano")):
    image = axis.imshow(matrix, cmap="viridis")
    axis.set_xticks(range(len(candidates)), candidates, rotation=25)
    axis.set_yticks(range(len(dynamics)), dynamics)
    axis.set_xlabel("operador candidato")
    axis.set_ylabel("dinámica verdadera")
    axis.set_title(title)
    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            axis.text(column_index, row_index, f"{matrix[row_index, column_index]:.3f}", ha="center", va="center", color="white" if matrix[row_index, column_index] > np.nanmedian(matrix) else "black")
    fig.colorbar(image, ax=axis, shrink=0.8)
fig.tight_layout()
plt.show()

fig, axis = plt.subplots(figsize=(8, 4.5))
for dynamics_name in dynamics:
    median_rollout = [np.median([row["rollout_errors"][dynamics_name][horizon] for row in results if row["dynamics"] == dynamics_name]) for horizon in rollout_horizons]
    axis.plot(rollout_horizons, median_rollout, marker="o", label=dynamics_name)
axis.set_xlabel("horizonte $h$")
axis.set_ylabel(r"$\lVert M^hA-AK^h\rVert_F/\lVert A\rVert_F$")
axis.set_title("Consistencia multi-step con el operador verdadero")
axis.legend()
fig.tight_layout()
plt.show()

In [ ]:
lines = ["## Lectura de los resultados", ""]
for dynamics_name in dynamics:
    values = summary[dynamics_name]
    lines.append(f"- **{dynamics_name}**: acción {values['correct_action_seeds']}/10, espectro {values['correct_spectrum_seeds']}/10 y rango completo {values['full_rank_seeds']}/10. El error correcto mediano es {values['median_correct_action_error']:.3f}; el margen frente al segundo candidato es {values['median_action_margin']:.3f}.")
lines.extend(["", f"**Conclusión primaria:** {'las tres condiciones superan el criterio predeclarado; esto apoya H3.' if global_operator_result else 'al menos una condición no alcanza el criterio predeclarado; H3 no recibe apoyo fuerte en esta receta.'}", "", "La loss total no participa en esta conclusión. Los gráficos deben mostrar una diagonal baja si el predictor sigue la dinámica: cada fila corresponde a los mismos marginales, pero un pairing temporal distinto. El rollout revela además si el error pequeño a un paso se mantiene o se acumula.", "", "Este notebook es desarrollo sobre validation. Aun con resultado positivo, el paso siguiente sería congelar la comparación completa antes de generar una nueva evaluación ciega."])
display(Markdown("\n".join(lines)))